In [0]:
%run ../control_framework/__init__ 

In [0]:
import time
from pyspark.sql import Row

In [0]:
dbutils.widgets.text("env", "dev", "env")
# dbutils.widgets.text("param_name", "ingestion", "param name")
dbutils.widgets.text("process_name", "finnhub_dp_daily_load", "process name")
dbutils.widgets.text("process_date", "20260716", "process date")
dbutils.widgets.text("run_type", "load", "run type")
dbutils.widgets.text("source", "m101_finnhub", "Source")
dbutils.widgets.text("target", "finnhub_dp", "Target")
dbutils.widgets.text("execution_type", "consumption_SADP", "execution type")

In [0]:
env = dbutils.widgets.get("env")
# param_name = dbutils.widgets.get("param_name")
process_name = dbutils.widgets.get("process_name")
process_date = dbutils.widgets.get("process_date")
process_date = int(process_date)
run_type = dbutils.widgets.get("run_type")
source = dbutils.widgets.get("source")
target = dbutils.widgets.get("target")
execution_type = dbutils.widgets.get("execution_type")
print(f"env: {env}")
# print(f"param_name: {param_name}")
print(f"process_name: {process_name}")
print(f"process_date: {process_date}")
print(f"run_type: {run_type}")
print(f"source: {source}")
print(f"target: {target}")
print(f"execution_type: {execution_type}")

In [0]:
try:
    comment = register_tgt(target, execution_type)
    insert_log(target, process_name, "target_register","target_register","target_register","target_register",process_date, 'success',comment)
except Exception as e:
    error_msg = str(e).replace("'", "''")
    insert_log(target, process_name, "target_register","target_register","target_register","target_register",process_date, 'failed', error_msg)
    print(f"Error in register_target: {e}")
    dbutils.notebook.exit(f"Error in register_target: {e}")

In [0]:
src_schema1 = f_get_src_schema(source)
src_tbl1 = 'finnhubb_daily_stock_price_ts'
# src_schema_master = f_get_src_schema('master')
# src_master_tb1 = 's_and_p_500_dev'
target_schema = f_get_tgt_schema(target)
target_tbl = 'finnhubb_daily_stock_price'
full_target_table_name = f"{target_schema}.{target_tbl}"
print(src_schema1)
print(src_tbl1)
# print(src_schema_master)
# print(src_master_tb1)
print(target_schema)
print(target_tbl)
print(full_target_table_name)

In [0]:
src_tbl_list = [{"source":source,"source_schema":src_schema1,"source_table": src_tbl1}]
tgt_tbl_list = [{"target":target,"target_schema":target_schema,"target_table": target_tbl,'holiday_ind': 'N','weekend_ind': 'N','calender':'','frequency': 'daily'}]

In [0]:
src_tgt_mapping_list = [
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'c','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'current_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'d','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'price_change','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'dp','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'price_change_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'h','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'day_high_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'l','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'day_low_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'o','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'opening_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'pc','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'previous_close_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'t','source_column_datatype':'long','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'quote_timestamp','target_column_datatype':'timestamp'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'t','source_column_datatype':'long','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'trade_date','target_column_datatype':'date'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'symbol','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'stock_symbol','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'process_date','source_column_datatype':'long','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'process_date','target_column_datatype':'long'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'r_source','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'r_source','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'hash_id','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'hash_id','target_column_datatype':'string'}
]

In [0]:
if run_type == 'load_metadata':
    status = run_load_metadata(src_tbl_list,tgt_tbl_list,target,process_name,src_tgt_mapping_list,process_date)
    dbutils.notebook.exit(status)


In [0]:
from pyspark.sql.types import StringType, DoubleType, LongType, StructType, StructField, TimestampType,DateType

schema_structure = {
    "stock_symbol": StringType(),
    "process_date": LongType(),
    "r_source": StringType(),
    "hash_id": StringType(),
    "current_price": DoubleType(),
    "price_change": DoubleType(),
    "day_high_price": DoubleType(),
    "day_low_price": DoubleType(),
    "opening_price": DoubleType(),
    "previous_close_price": DoubleType(),
    "quote_timestamp": TimestampType(),
    "trade_date":DateType(),
    "price_change_percent": DoubleType(),
    "r_target": StringType(),
}

In [0]:
delta_logic = f"""process_date = {process_date}"""

In [0]:
if run_type == 'full':
    create_tgt_tbl_log = create_target_table(full_target_table_name,schema_structure)
    if  create_tgt_tbl_log == f"Table {target_schema}.{target_tbl} created successfully":
        process_status = 'success'
    else:
        process_status = 'failed'
    log = str(create_tgt_tbl_log).replace("'", "''")
    insert_log (target, process_name, 'create_tgt_tbl','create_tgt_tbl', target_schema, target_tbl, process_date,process_status, log)

    delta_logic = f"""process_date <= {process_date}"""

In [0]:
column_fetch = f"""
pc as previous_close_price,
dp as price_change_percent,
process_date,
symbol as stock_symbol,
r_source,
h as day_high_price,
l as day_low_price,
o as opening_price,
c as current_price,
d as price_change,
CAST(from_unixtime(t) AS TIMESTAMP) AS quote_timestamp,
TO_DATE(from_unixtime(t)) AS trade_date,
hash_id
"""

In [0]:
v_sel_main1 = f"""with s as (
  select *, row_number() over (partition by hash_id order by process_date desc) as rn, '{target}' as r_target
  from {src_schema1}.{src_tbl1}
  where {delta_logic}
),
src as (
  select {column_fetch} from s where rn = 1
),
tgt as (
  select * from {target_schema}.{target_tbl}
  where r_target = '{target}'
),
main as (
  select src.*
  from src
  left anti join tgt
  on src.hash_id = tgt.hash_id
)
select *,'{target}' as r_target  from main
"""
print(v_sel_main1)

In [0]:
insert_table_query = f"""insert into {target_schema}.{target_tbl}
(previous_close_price,
  price_change_percent,
  process_date,
  stock_symbol,
  r_source,
  day_high_price,
  day_low_price,
  opening_price,
  current_price,
  price_change,
  quote_timestamp,
  trade_date,
  hash_id,
  r_target)
 {v_sel_main1}"""
print(insert_table_query)

In [0]:
if run_type == "rollback":
    try:
        comment = rollback_run(tgt_tbl_list,process_date)
        insert_log(target, process_name, 'rollback', 'rollback', target_schema, target_tbl, process_date, 'success', str(comment).replace("'", "''"))
    except Exception as e:
        error_msg = str(e).replace("'", "''")
        insert_log(target, process_name, 'rollback', 'rollback', target_schema, target_tbl, process_date, 'failed', error_msg)
        comment = f"Error in rollback: {e}"
    dbutils.notebook.exit(f"{comment}")

In [0]:
if run_type == 'full' or run_type == 'load':
    try:
        # data_list_val = data_list(symbol_list)
        # insert_ingestion_table_query(data_list_val, target_schema, target_tbl)
        # data_df = get_data(data_list_val,schema_structure)
        # anomoly_df = validate_and_store_anomalies(data_df,schema_structure,full_target_table_name)
        # clean_df = remove_anomalies_by_hash_id(data_df, anomoly_df)
        spark.sql(f"""{insert_table_query}""")
        insert_log(target, process_name, 'data_consumption', 'data_consumption', target_schema, target_tbl, process_date, 'success', 'Data consumption completed')
        e = "success"
    except Exception as e:
        error_msg = str(e).replace("'", "''")
        insert_log(target, process_name, 'data_consumption', 'data_consumption', target_schema, target_tbl, process_date, 'failed', error_msg)
        e = f"Error in data_consumption: {e}"
    dbutils.notebook.exit(f"{e}")